# Bitcoin Zaman Serisi Tahmini — RNN, LSTM, GRU ve Finansal Doğrulama

Bu notebook, Bitcoin saatlik verisi üzerinde tek değişkenli ve çok değişkenli modelleri karşılaştırır. Amaç yalnızca düşük hata üretmek değil; **adil zaman serisi değerlendirmesi, güçlü baseline'lar, gerçekçi backtest ve tekrarlanabilir deney** kurmaktır.

**Önemli:** Bu çalışma eğitim/araştırma amaçlıdır; yatırım tavsiyesi değildir. Geçmiş test performansı gelecekteki getiriyi garanti etmez. Veri kaynağı ve sürümü değiştikçe sonuçlar da değişebilir.

## Yol haritası
1. Ortam ve veri
2. Veri temizliği ve saatlik seri
3. Tahmin problemi ve kronolojik bölme
4. Naive baseline
5. SimpleRNN / LSTM / GRU
6. Sonuçların eleştirel değerlendirilmesi
7. Log getiri modellemesi
8. Log getiri sonuçlarının değerlendirilmesi
9. Çok değişkenli GRU ve özellik mühendisliği
10. Çok değişkenli sonuçların analizi
11. Backtest ve portföy metrikleri
12. Walk-forward doğrulama, benchmark'lar ve kalite kapıları

Bölüm geçişlerinde, elde edilen bulgunun neden bir sonraki deneyi gerektirdiği açıklanır. Sonuç metinleri mümkün olduğunca çalıştırılan hücrelerden üretilir; sabit ve doğrulanmamış başarı iddiaları kullanılmaz.

## 1. Ortamı hazırla

Colab'da **Runtime → Run all** kullanabilir veya hücreleri sırayla çalıştırabilirsin. TensorFlow genellikle Colab'da kurulu gelir; bu hücre eksikse yükler.

In [ ]:
# Gerekli kütüphaneleri yüklemek ve sürümleri kontrol etmek için sistem araçlarını içe aktar
import sys, subprocess, importlib.util

# Eğer 'tensorflow' kütüphanesi kurulu değilse, pip kullanarak sessizce yükle
if importlib.util.find_spec("tensorflow") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
# Eğer 'kaggle' kütüphanesi kurulu değilse, pip kullanarak sessizce yükle
if importlib.util.find_spec("kaggle") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
# KaggleHub, kimlik doğrulaması gerektirmeyen veri indirme akışında kullanılır
if importlib.util.find_spec("kagglehub") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])

# Dosya işlemleri, veri işleme, görselleştirme ve yapay öğrenme kütüphanelerini içe aktar
import os, glob, json, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow import keras
from tensorflow.keras import layers

# Tekrarlanabilir sonuçlar elde etmek için rastgelelik durumunu sabitle
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Sistemde kurulu olan TensorFlow ve Pandas sürümlerini ve GPU desteğini yazdır
print("TensorFlow:", tf.__version__)
print("Pandas:", pd.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Veriyi al

Ana akışta KaggleHub ile veri indiriyoruz; bu yöntem için ayrıca `kaggle.json` yükleme hücresi zorunlu değildir. Alternatif olarak CSV'yi manuel yükleyebilirsin. Kaggle erişimi ortamında kimlik doğrulaması isterse aşağıdaki indirme hücresinde gösterilen hata mesajını izle.

**Güvenlik:** API token'ını notebook'a düz metin olarak yazma ve paylaşma.

### Alternatif: CSV'yi kendin yükle

KaggleHub ile indirme çalışmazsa CSV dosyasını Colab'ın sol panelindeki dosya yükleme aracıyla `/content` dizinine yükleyebilirsin. Dosyada `Timestamp` ve `Close` sütunları bulunmalı. Sonraki hücreler indirilen klasörleri arar; manuel yükleme yaparsan aşağıdaki hücrede `csv_files` listesini yüklediğin dosya yoluyla tanımla.

### KaggleHub kurulumu

Ortam hazırlama hücresi KaggleHub paketini kurar. CSV'yi manuel yüklediysen de dosya arama hücresi `/content` klasörünü kontrol eder.

### Alternatif: CSV dosyasını elle seçme

İndirme başarısız olursa CSV'yi Colab'ın sol panelindeki dosya yükleme aracıyla `/content` içine yükle. Dosyada `Timestamp` ve `Close` sütunları bulunmalı. Aşağıdaki hücreler bu konumu da tarar.

In [ ]:
# KaggleHub ile veri setini indir
import kagglehub
import os

try:
    path = kagglehub.dataset_download("mczielinski/bitcoin-historical-data")
    print("Dataset path:", path)
    print("Files:", os.listdir(path))
except Exception as exc:
    print("KaggleHub indirmesi başarısız oldu:", repr(exc))
    print("Alternatif: CSV dosyasını Colab'da /content içine yükleyip devam et.")
    path = None

In [ ]:
# İndirilen klasördeki CSV dosyalarını bul; indirme başarısızsa /content içinde ara.
import glob
import pandas as pd

csv_files = []
if path:
    csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
if not csv_files:
    csv_files = glob.glob("/content/**/*.csv", recursive=True)
print("Bulunan CSV dosyaları:", csv_files[:10])

In [ ]:
# Dosyanın yapısını kontrol etmek için yalnızca ilk 5 satırı oku; büyük CSV'yi belleğe tamamen alma.
if not csv_files:
    # Colab'a manuel yüklenen CSV'leri de ara.
    csv_files = glob.glob("/content/**/*.csv", recursive=True)
if not csv_files:
    raise FileNotFoundError("CSV bulunamadı. KaggleHub indirmesini kontrol et veya CSV'yi /content içine yükle.")

CSV_PATH = csv_files[0]
df_sample = pd.read_csv(CSV_PATH, nrows=5, low_memory=False)
print("Seçilen CSV:", CSV_PATH)
print("Örnek boyut:", df_sample.shape)
print("Sütunlar:", df_sample.columns.tolist())
print("Veri tipleri:")
print(df_sample.dtypes)
print("İlk 5 satır:")
display(df_sample.head())

**Bellek notu:** Dosyanın örnek satırları yukarıda kontrol edildi. Büyük CSV dosyasını ikinci kez tamamen belleğe yüklemiyoruz; sonraki bölüm gerekli sütunları seçerek veriyi işler.

### Geçiş — Ham veriden saatlik seriye
Dosya erişilebilir olsa da ham kayıtlar doğrudan modele verilemez. Önce zaman damgalarını, sayısal alanları, yinelenen kayıtları ve saatlik OHLCV toplulaştırmasını kontrol edeceğiz.

## 3. CSV'yi temizle ve saatlik seriye dönüştür

Kaggle veri sürümlerinde hacim sütununun adı değişebildiği için kod olası sütun adlarını kontrol eder. `Timestamp` Unix saniyesi veya tarih metni olabilir. Saatlik mumlar için Open=ilk, High=maksimum, Low=minimum, Close=son, Volume=toplam olarak gruplanır.

In [ ]:
# Dakikalık veriyi okuyan, temizleyen ve saatlik OHLCV verisine dönüştüren fonksiyon
def load_hourly_ohlcv(csv_path):
    # Bellek tasarrufu için sadece sütun isimlerini (0 satır) oku
    header = pd.read_csv(csv_path, nrows=0).columns.tolist()
    # Sütun isimlerindeki boşlukları temizle ve hepsini küçük harfe çevir
    normalized = {str(c).strip().lower(): c for c in header}

    # İstenen sütun adının veri setinde hangi isimle yer aldığını bulan iç fonksiyon
    def find_col(candidates, required=False):
        for candidate in candidates:
            if candidate in normalized:
                return normalized[candidate]
        if required:
            raise ValueError(f"Gerekli sütun bulunamadı. Mevcut sütunlar: {header}")
        return None

    # Zaman damgası ve Kapanış fiyatı gibi zorunlu sütunları tespit et
    ts_col = find_col(["timestamp", "date", "datetime", "time"], required=True)
    open_col = find_col(["open"])
    high_col = find_col(["high"])
    low_col = find_col(["low"])
    close_col = find_col(["close", "close_price"], required=True)
    volume_col = find_col(["volume", "volume_(btc)", "volume_btc", "volume_(currency)", "volume_currency"])

    # Sadece tespit edilen gerekli sütunları içeren listeyi oluştur
    usecols = list(dict.fromkeys([c for c in [ts_col, open_col, high_col, low_col, close_col, volume_col] if c is not None]))
    # CSV dosyasından sadece bu sütunları okuyarak veri çerçevesi oluştur
    frame = pd.read_csv(csv_path, usecols=usecols, low_memory=False)

    # Sütun isimlerini standartlaştırmak için yeniden adlandır
    rename = {ts_col: "Timestamp", close_col: "Close"}
    if open_col: rename[open_col] = "Open"
    if high_col: rename[high_col] = "High"
    if low_col: rename[low_col] = "Low"
    if volume_col: rename[volume_col] = "Volume"
    frame = frame.rename(columns=rename)

    # Zaman sütununu kontrol et ve datetime formatına dönüştür
    ts = frame["Timestamp"]
    if pd.api.types.is_numeric_dtype(ts):
        # Unix zaman damgasını UTC tarih formatına çevir
        frame["Timestamp"] = pd.to_datetime(ts, unit="s", utc=True, errors="coerce")
    else:
        # Metinsel tarihleri dönüştür
        parsed = pd.to_datetime(ts, utc=True, errors="coerce")
        if parsed.notna().mean() < 0.8:
            numeric_ts = pd.to_numeric(ts, errors="coerce")
            parsed_numeric = pd.to_datetime(numeric_ts, unit="s", utc=True, errors="coerce")
            if parsed_numeric.notna().mean() > parsed.notna().mean():
                parsed = parsed_numeric
        frame["Timestamp"] = parsed

    # Sayısal olması gereken tüm sütunları sayısal veri tipine zorla dönüştür
    for col in ["Open", "High", "Low", "Close", "Volume"]:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors="coerce")

    # Tarihi veya Kapanış fiyatı olmayan satırları veri setinden temizle
    frame = frame.dropna(subset=["Timestamp", "Close"])
    # Kapanış fiyatı sıfırdan büyük olan geçerli satırları filtrele
    frame = frame[frame["Close"] > 0]
    # Tarihe göre sırala ve mükerrer (aynı ana ait birden fazla) kayıtları kaldır
    frame = frame.sort_values("Timestamp").drop_duplicates("Timestamp", keep="last")
    # Zaman damgasını veri setinin ana indeksi (satır etiketi) yap
    frame = frame.set_index("Timestamp")

    # Saatlik gruplama yaparken her sütun için hangi yöntemin kullanılacağını belirle
    agg = {}
    if "Open" in frame: agg["Open"] = "first"
    if "High" in frame: agg["High"] = "max"
    if "Low" in frame: agg["Low"] = "min"
    agg["Close"] = "last"
    if "Volume" in frame: agg["Volume"] = "sum"

    # Dakikalık veriyi saatlik mumlara grupla ve kurallara göre birleştir
    hourly = frame.resample("1h").agg(agg)
    # Fiyatı boş kalan saatleri temizle
    hourly = hourly.dropna(subset=["Close"])
    return hourly

# Sistemde yüklü csv dosyası varsa yolunu belirle, yoksa hata fırlat
if 'csv_files' in globals() and csv_files:
    CSV_PATH = csv_files[0]
else:
    raise NameError("CSV_PATH tanımlı değil ve indirilen csv_files listesi bulunamadı.")

# Veriyi saatlik hale getir
hourly = load_hourly_ohlcv(CSV_PATH)
# Veri miktarının analiz için yeterli olduğundan emin ol
if len(hourly) < 24 * 30:
    raise ValueError(f"Model için beklenenden az saatlik kayıt var: {len(hourly)}")

# Dönüştürülmüş saatlik verinin özet bilgilerini ve ilk/son satırlarını ekrana yazdır
print("Saatlik kayıt sayısı:", f"{len(hourly):,}")
print("Başlangıç:", hourly.index.min())
print("Bitiş:", hourly.index.max())
display(hourly.head())
display(hourly.tail())

In [ ]:
# Grafik çizimi için 14'e 4 boyutlarında bir figür oluştur
fig, ax = plt.subplots(figsize=(14, 4))
# Saatlik kapanış fiyatlarını zaman ekseninde çizgi grafiği olarak çizdir
hourly["Close"].plot(ax=ax, title="Bitcoin — Saatlik Kapanış Fiyatı")
# Grafik eksen etiketlerini tanımla
ax.set_ylabel("Close price (USD)")
ax.set_xlabel("Zaman (UTC)")
# Grafiği ekranda göster
plt.show()

# Verideki eksik kapanış fiyatlarının toplam sayısını yazdır
print("Eksik Close:", int(hourly["Close"].isna().sum()))
# Aynı tarihe ait mükerrer kayıt olup olmadığını kontrol et ve yazdır
print("Tekrarlı zaman damgası:", int(hourly.index.duplicated().sum()))
# Kapanış fiyatlarının ortalama, standart sapma, min ve maks gibi istatistiksel özetini göster
print("Close özeti:")
display(hourly["Close"].describe())

### Geçiş — Keşif analizinden problem tanımına
Saatlik seri oluşturulduktan sonra tahmin ufkunu ve geçmiş pencere uzunluğunu sabitliyoruz. Kronolojik bölme, gelecekteki bilgiyi eğitim sürecinden uzak tutmak için zorunludur.

## 4. Tahmin problemini tanımla

**Hedef:** Son 24 saatlik kapanış fiyatından bir sonraki saatin kapanış fiyatını tahmin etmek.

- `LOOKBACK = 24`: model her tahminde önceki 24 saati görür.
- Veriyi kronolojik olarak %80 eğitim, %20 test şeklinde böleriz.
- Ölçekleyiciyi yalnızca eğitim verisine uydururuz; böylece gelecek test verisi eğitim ön işlemesine sızmaz.
- Test pencereleri, test döneminin ilk tahmini için eğitim döneminin son 24 saatini bağlam olarak kullanır.

In [ ]:
# Geriye dönük bakılacak saat sayısını ve eğitim verisi oranını tanımla
LOOKBACK = 24
TRAIN_RATIO = 0.80

# Kapanış fiyatlarını float32 tipinde tek sütunluk bir matrise dönüştür
close_values = hourly["Close"].to_numpy(dtype=np.float32).reshape(-1, 1)
# Eğitim verisinin nerede biteceğini hesapla
split_idx = int(len(close_values) * TRAIN_RATIO)
# Veri miktarının pencereler oluşturmak için yeterli olup olmadığını kontrol et
if split_idx <= LOOKBACK or len(close_values) - split_idx < 24:
    raise ValueError("Train/test bölümü çok küçük. Daha uzun bir veri seti kullan.")

# Verileri 0-1 arasına normalize etmek için ölçekleyiciyi tanımla
scaler = MinMaxScaler()
# Ölçekleyiciyi sadece eğitim verisi üzerinden eğit (veri sızıntısını önle)
scaler.fit(close_values[:split_idx])
# Tüm veri setini bu ölçekleyici ile normalize et
scaled_all = scaler.transform(close_values).astype(np.float32)

# Ölçeklenmiş veriyi eğitim ve test (bağlam dahil) parçalarına ayır
train_scaled = scaled_all[:split_idx]
test_context_scaled = scaled_all[split_idx - LOOKBACK:]

# Zaman serisini RNN modelleri için gözetimli öğrenme pencerelerine (X ve y) dönüştüren fonksiyon
def make_windows(data, lookback=LOOKBACK):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback:i])
        y.append(data[i, 0])
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32).reshape(-1, 1)

# Eğitim ve test setleri için girdi (X) ve hedef (y) pencerelerini oluştur
X_train_all, y_train_all = make_windows(train_scaled)
# Test seti pencerelerini oluştur
X_test, y_test = make_windows(test_context_scaled)

# Aşırı öğrenmeyi engellemek için eğitim verisinin son %10'unu doğrulama seti olarak ayır
val_idx = int(len(X_train_all) * 0.90)
# Eğitim pencerelerini tanımla
X_train, y_train = X_train_all[:val_idx], y_train_all[:val_idx]
# Doğrulama pencerelerini tanımla
X_val, y_val = X_train_all[val_idx:], y_train_all[val_idx:]

# Oluşturulan pencerelerin boyutlarını ekrana yazdır
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "y_test: ", y_test.shape)
# Matris boyutlarının RNN standartlarına uygunluğunu doğrula
assert X_train.ndim == 3 and X_train.shape[1:] == (LOOKBACK, 1)
# Girdi ve hedef veri uzunluklarının eşit olduğunu kontrol et
assert len(X_test) == len(y_test)

### Geçiş — Problemi tanımlamaktan ölçülebilir referansa
Bir sinir ağının faydasını anlayabilmek için önce basit bir referans gerekir. Son kapanışı bir sonraki kapanış olarak tahmin etmek, rastgele yürüyüşe yakın piyasalarda güçlü bir başlangıç benchmark’ıdır.

## 5. Baseline: son fiyatı tahmin olarak kullan

Finansal zaman serilerinde sinir ağı, basit bir baseline'dan daha iyi olmalıdır. Baseline, bir sonraki saatin kapanışını mevcut son kapanışa eşit kabul eder. Modellerin MAE/RMSE değerlerini bu sonuçla karşılaştıracağız.

In [ ]:
# Gerçek test değerlerini orijinal fiyat ölçeğine geri dönüştür
y_test_actual = scaler.inverse_transform(y_test).ravel()
# Test girdisindeki en son saatlik kapanışı (t-1 anını) doğrudan tahmin kabul et
naive_scaled = X_test[:, -1, 0].reshape(-1, 1)
# Bu tahmini orijinal fiyat ölçeğine geri çevir
y_naive = scaler.inverse_transform(naive_scaled).ravel()

# Son fiyatı alan baseline modelinin Ortalama Mutlak Hatasını (MAE) hesapla
baseline_mae = mean_absolute_error(y_test_actual, y_naive)
# Son fiyatı alan baseline modelinin Kök Ortalama Kare Hatasını (RMSE) hesapla
baseline_rmse = np.sqrt(mean_squared_error(y_test_actual, y_naive))
# Hesaplanan baseline hata metriklerini dolar cinsinden ekrana yazdır
print(f"Baseline MAE : {baseline_mae:,.2f} USD")
print(f"Baseline RMSE: {baseline_rmse:,.2f} USD")

### Geçiş — Baseline’dan öğrenilebilir modellere
Baseline artık ölçülebilir. Aynı veri pencereleri ve aynı test dönemi üzerinde SimpleRNN, LSTM ve GRU eğiterek daha karmaşık modellerin gerçek katkısını sınayacağız.

## 6. SimpleRNN, LSTM ve GRU modellerini eğit

Adil karşılaştırma için aynı pencere ve aynı train/validation/test bölümleri kullanılır. Erken durdurma, validation kaybı iyileşmediğinde eğitimi durdurur.

In [ ]:
# İstenen türe (SimpleRNN, LSTM, GRU) göre Keras yapay sinir ağı modelini kuran fonksiyon
def build_model(kind):
    recurrent_layers = {
        "SimpleRNN": layers.SimpleRNN(32, activation="tanh"),
        "LSTM": layers.LSTM(32),
        "GRU": layers.GRU(32),
    }
    if kind not in recurrent_layers:
        raise ValueError(f"Bilinmeyen model: {kind}")
    # Giriş, yinelemeli (RNN), ara (Dense) ve çıkış katmanlarını uç uca ekle
    model = keras.Sequential([
        keras.Input(shape=(LOOKBACK, 1)),
        recurrent_layers[kind],
        layers.Dense(16, activation="relu"),
        layers.Dense(1)
    ], name=kind)
    # Modeli Adam optimizasyon algoritması ve MSE hata fonksiyonu ile derle
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse", metrics=["mae"])
    return model

# Sonuçları, eğitilen modelleri ve geçmiş grafik verilerini tutacak yapıları tanımla
results = []
models = {}
histories = {}
EPOCHS = 12
BATCH_SIZE = 128

# Her bir sinir ağı mimarisini döngü içerisinde sırayla eğit
for model_name in ["SimpleRNN", "LSTM", "GRU"]:
    print(f"\n===== {model_name} eğitiliyor =====")
    # Bellekte eski model kalıntıları kalmaması için Keras oturumunu temizle
    tf.keras.backend.clear_session()
    # Modeli oluştur
    model = build_model(model_name)
    # Doğrulama kaybı 3 epoch boyunca düşmezse eğitimi durduracak erken durdurma kuralını tanımla
    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True, verbose=1
    )
    # Modeli eğitim ve doğrulama verileri ile eğit
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=False,
        callbacks=[early_stop],
        verbose=1
    )
    # Eğitilen modeli test seti üzerinde çalıştırarak tahminleri al
    pred_scaled = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
    # Normalize tahminleri gerçek dolar fiyatı ölçeğine dönüştür
    y_pred = scaler.inverse_transform(pred_scaled).ravel()
    # Test seti için MAE ve RMSE hata metriklerini hesapla
    mae = mean_absolute_error(y_test_actual, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))
    # Sonuçları listeye kaydet
    results.append({"Model": model_name, "MAE_USD": mae, "RMSE_USD": rmse,
                    "Epochs": len(history.history["loss"])})
    # Modeli ve eğitim geçmişini sakla
    models[model_name] = model
    histories[model_name] = history
    print(f"{model_name} | MAE={mae:,.2f} USD | RMSE={rmse:,.2f} USD")

# Sonuçları içeren bir veri çerçevesi oluştur ve RMSE değerine göre en iyiden en kötüye sırala
results_df = pd.DataFrame(results).sort_values("RMSE_USD").reset_index(drop=True)
display(results_df)
# Karşılaştırma için en başta hesaplanan baseline hata değerlerini yazdır
print("Baseline MAE/RMSE:", round(baseline_mae, 2), "/", round(baseline_rmse, 2))

In [ ]:
# 10'a 4 boyutlarında doğrulama kaybı grafiği için alan oluştur
fig, ax = plt.subplots(figsize=(10, 4))
# Her modelin doğrulama kayıplarını epoch bazında çizdir
for name, history in histories.items():
    ax.plot(history.history["val_loss"], label=f"{name} val_loss")
# Grafik başlığını ve eksen isimlerini tanımla
ax.set_title("Validation Loss (ölçeklenmiş hedef üzerinde MSE)")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE")
ax.legend()
plt.show()

# En düşük test RMSE değerine sahip olan modelin adını seç
best_name = results_df.iloc[0]["Model"]
# En başarılı modeli çek
best_model = models[best_name]
# Bu model ile test seti üzerinde tekrar tahmin yap
best_pred_scaled = best_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
# Tahminleri orijinal dolar ölçeğine geri dönüştür
best_pred = scaler.inverse_transform(best_pred_scaled).ravel()

# 14'e 5 boyutlarında tahmin karşılaştırma grafiği oluştur
fig, ax = plt.subplots(figsize=(14, 5))
# Gerçek fiyatları, en iyi model tahminlerini ve baseline tahminlerini ilk 300 saat için çizdir
ax.plot(y_test_actual[:300], label="Gerçek kapanış", linewidth=1.5)
ax.plot(best_pred[:300], label=f"Tahmin ({best_name})", linewidth=1.2)
ax.plot(y_naive[:300], label="Baseline: son fiyat", alpha=0.7, linewidth=1)
# Grafik detaylarını tanımla ve ekranda göster
ax.set_title("Test dönemi — ilk 300 saat")
ax.set_xlabel("Test saatleri")
ax.set_ylabel("BTC kapanış fiyatı (USD)")
ax.legend()
plt.show()
# En başarılı olan modelin ismini ekrana yazdır
print("En düşük test RMSE sonucu:", best_name)

## 7. Sonuçların Yorumu ve Bir Sonraki Adım

Aşağıdaki metrikler, önceki eğitim hücresinin gerçek çıktılarından gelmelidir; veri seti değişebildiği için burada sabit RMSE/MAE değerleri yazmıyoruz. `results_df` ile baseline tablosunu birlikte inceleyin.

**Yol ayrımı:** Eğer fiyat seviyeli RNN modelleri naive baseline'ı geçemiyorsa, karmaşıklık eklemek yerine hedef değişkenini yeniden tanımlamak mantıklıdır. Bu nedenle sonraki bölümde log getirilerle eğitim yapacağız. Bununla birlikte, log getirilerin durağan olduğu veya mutlaka daha iyi sonuç vereceği varsayılmayacaktır.

### Geçiş — Fiyat seviyesinden getiriye
İlk deneyde fiyat seviyesinin tahmin edilmesi zorlu bir hedef olabilir. Bu nedenle hedefi log getiriye çeviriyoruz; bu dönüşümün faydalı olup olmadığı yalnızca aynı test dönemindeki sonuçlarla anlaşılabilir.

## 8. İyileştirme: Log Getirilerle Modelleme

İlk deneyde ham fiyat seviyeleriyle eğitilen RNN ailesinin naive son-fiyat baseline'ını geçip geçmediğini ölçtük. Fiyat seviyesinin ölçeği ve trendi öğrenmeyi zorlaştırabileceğinden şimdi hedefi log getiriye dönüştürüyoruz:

\[r_t = \ln(C_t/C_{t-1})\]

Log getiri kullanmak seriyi otomatik olarak durağan yapmaz ve tahmin edilebilirliği garanti etmez; bu, test edilmesi gereken bir hipotezdir. Bir sonraki bölümde getirileri aynı zaman indeksindeki gerçek fiyatlara dönüştürüp baseline ile adil biçimde karşılaştıracağız.

In [ ]:
# 1. Log Getirileri Hesaplama
# Saatlik kapanış fiyatlarının logaritmik farkını alarak getiri serisini oluştur
hourly['Log_Return'] = np.log(hourly['Close'] / hourly['Close'].shift(1))
# İlk satır boş (NaN) kalacağı için o satırı temizle
hourly_clean = hourly.dropna(subset=['Log_Return'])

# Getiri değerlerini matris biçimine getir
returns_values = hourly_clean['Log_Return'].to_numpy(dtype=np.float32).reshape(-1, 1)

# Bölme indeksini hesapla
split_idx_ret = int(len(returns_values) * TRAIN_RATIO)

# Getiriler zaten doğal olarak -1 ile 1 arasında küçük değerler olduğu için
# MinMaxScaler kullanabiliriz ancak veri sızıntısını önlemek için yine sadece train üzerinden fit ediyoruz
scaler_ret = MinMaxScaler(feature_range=(-1, 1))
scaler_ret.fit(returns_values[:split_idx_ret])
scaled_returns = scaler_ret.transform(returns_values).astype(np.float32)

# Eğitim ve test setlerini oluştur
train_ret_scaled = scaled_returns[:split_idx_ret]
test_ret_context_scaled = scaled_returns[split_idx_ret - LOOKBACK:]

# Pencereleri oluştur
X_train_ret_all, y_train_ret_all = make_windows(train_ret_scaled)
X_test_ret, y_test_ret = make_windows(test_ret_context_scaled)

# Validation setini ayır
val_idx_ret = int(len(X_train_ret_all) * 0.90)
X_train_ret, y_train_ret = X_train_ret_all[:val_idx_ret], y_train_ret_all[:val_idx_ret]
X_val_ret, y_val_ret = X_train_ret_all[val_idx_ret:], y_train_ret_all[val_idx_ret:]

print("Log Getiri X_train boyutu:", X_train_ret.shape)
print("Log Getiri X_test boyutu:", X_test_ret.shape)

In [ ]:
# 2. Modelleri Log Getirilerle Yeniden Eğitme
results_ret = []
models_ret = {}
histories_ret = {}

for model_name in ["SimpleRNN", "LSTM", "GRU"]:
    print(f"\n===== {model_name} (Log Getiri) eğitiliyor =====")
    tf.keras.backend.clear_session()

    # Aynı mimariyi log getiri boyutuna uygun olarak kur
    model = build_model(model_name)

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True, verbose=1
    )

    history = model.fit(
        X_train_ret, y_train_ret,
        validation_data=(X_val_ret, y_val_ret),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=False,
        callbacks=[early_stop],
        verbose=1
    )

    # Test tahmini gerçekleştir
    pred_ret_scaled = model.predict(X_test_ret, batch_size=BATCH_SIZE, verbose=0)
    # Normalizasyonu geri al
    pred_ret = scaler_ret.inverse_transform(pred_ret_scaled).ravel()

    # Tahmin edilen getirileri sakla
    models_ret[model_name] = pred_ret
    histories_ret[model_name] = history

In [ ]:
# 3. Log getirileri aynı timestamp'teki fiyatlarla eşleştir
# Log getiriler hourly_clean indeksinde tutulur; böylece sabit kayıt sayısına ihtiyaç kalmaz.
test_return_index = hourly_clean.index[split_idx_ret:]
y_test_actual_ret_aligned = hourly.loc[test_return_index, "Close"].to_numpy(dtype=np.float64)
y_test_prev_actual_aligned = hourly["Close"].shift(1).reindex(test_return_index).to_numpy(dtype=np.float64)

assert len(y_test_prev_actual_aligned) == len(y_test_actual_ret_aligned) == len(next(iter(models_ret.values())))
assert np.isfinite(y_test_prev_actual_aligned).all() and np.isfinite(y_test_actual_ret_aligned).all()

results_ret = []
for model_name, pred_returns in models_ret.items():
    price_pred = y_test_prev_actual_aligned * np.exp(pred_returns)
    mae_usd = mean_absolute_error(y_test_actual_ret_aligned, price_pred)
    rmse_usd = np.sqrt(mean_squared_error(y_test_actual_ret_aligned, price_pred))
    results_ret.append({"Model": f"{model_name} (LogRet)", "MAE_USD": mae_usd, "RMSE_USD": rmse_usd})

results_ret_df = pd.DataFrame(results_ret).sort_values("RMSE_USD").reset_index(drop=True)
print("Log getiri modellerinin gerçek fiyat ölçeğindeki sonuçları:")
display(results_ret_df)
print(f"Naive fiyat baseline RMSE: {baseline_rmse:,.2f} USD")

In [ ]:
# 4. En iyi log-getiri modelini aynı timestamp'lerde görselleştir
best_new_model_name = results_ret_df.iloc[0]["Model"].split()[0]
best_new_returns = models_ret[best_new_model_name]
best_new_price_pred = y_test_prev_actual_aligned * np.exp(best_new_returns)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_test_actual_ret_aligned[:300], label="Gerçek kapanış", linewidth=1.5)
ax.plot(best_new_price_pred[:300], label=f"{best_new_model_name} (LogRet)", linewidth=1.2, linestyle="--")
ax.plot(y_naive[:len(best_new_price_pred)][:300], label="Naive baseline", alpha=0.6, linewidth=1)
ax.set_title("Log Getiri Modeli ve Naive Baseline — İlk 300 Test Saati")
ax.set_xlabel("Test saati")
ax.set_ylabel("BTC kapanış fiyatı (USD)")
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()

## 9. Log Getiri Deneyinin Yorumu

Sonuç tablosunu çalıştırma sırasında üretilen `results_ret_df` ve `baseline_rmse` değerleri üzerinden yorumlayın. Fiyat tahminleri, her test zamanının bir önceki **gerçek** kapanışı ile aynı timestamp'teki tahmin edilen log getirinin birleştirilmesiyle hesaplanır.

**Yol haritası:** Log getiri hedefi baseline'a yaklaşsa bile tek değişkenli modelin sınırlı bilgi taşıması mümkündür. Bu nedenle sonraki bölümde volatilite, RSI ve fiyatın hareketli ortalamaya oranını ekleyeceğiz. Ek özelliklerin katkısı önceden varsayılmayacak; aynı test dönemi ve baseline üzerinde ölçülecektir.

### Geçiş — Tek değişkenden çok değişkene
Log getirilerle hedefi yeniden tanımladık. Şimdi geçmiş getiriden türetilen volatilite, RSI ve fiyat/SMA oranını ekleyerek ilave özelliklerin genelleme başarısını artırıp artırmadığını test edeceğiz.

## 10. İleri Düzey İyileştirme: Çok Değişkenli (Multivariate) GRU ve Özellik Mühendisliği

Sadece geçmiş fiyat getirilerine bakmak yerine, finansal analizlerde yaygın olarak kullanılan teknik göstergeleri modele ek girdi olarak besleyeceğiz. Bu, modelin trend değişimlerini, aşırı alım/satım bölgelerini ve piyasa oynaklığını (volatilitesini) anlamasını kolaylaştıracaktır.

### Eklenecek Teknik Özellikler:
1. **Log Return Volatility (5 Saatlik):** Getirilerin hareketli standart sapması. Oynaklık bilgisini taşır.
2. **RSI (Göreceli Güç Endeksi - 14 Saatlik):** Momentum bilgisini ve aşırı alım/satım bölgelerini temsil eder.
3. **Fiyat/Ortalama Oranı (24 Saatlik SMA):** Mevcut fiyatın 24 saatlik ortalamaya göre nerede olduğunu gösterir.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Özellik Mühendisliği Uygulama
df_features = hourly.copy()

# Log Getiriyi zaten hesaplamıştık
df_features['Log_Return'] = np.log(df_features['Close'] / df_features['Close'].shift(1))

# Özellik A: Log Return Volatility (5 saatlik pencere)
df_features['Volatility_5h'] = df_features['Log_Return'].rolling(window=5).std()

# Özellik B: Basit Hareketli Ortalama Oranı (Mevcut Close / 24h SMA)
df_features['SMA_24'] = df_features['Close'].rolling(window=24).mean()
df_features['Close_to_SMA'] = df_features['Close'] / df_features['SMA_24']

# Özellik C: RSI (Göreceli Güç Endeksi - 14 Saatlik)
def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / (loss + 1e-9)
    return 100 - (100 / (1 + rs))

df_features['RSI_14'] = calculate_rsi(df_features['Close'], 14)

# NaN içeren ilk satırları temizleyelim
df_features = df_features.dropna(subset=['Log_Return', 'Volatility_5h', 'Close_to_SMA', 'RSI_14'])

# Kullanacağımız girdiler ve tahmin hedefi
feature_cols = ['Log_Return', 'Volatility_5h', 'Close_to_SMA', 'RSI_14']
target_col = 'Log_Return'

# Veriyi matrise çevirelim
X_raw = df_features[feature_cols].to_numpy(dtype=np.float32)
y_raw = df_features[target_col].to_numpy(dtype=np.float32).reshape(-1, 1)

# Kronolojik bölme (Train/Test %80 - %20)
split_idx_adv = int(len(df_features) * 0.80)

# Veri sızıntısını önlemek için ölçeklendiriciyi sadece train verisinde eğitiyoruz
scaler_X = MinMaxScaler(feature_range=(-1, 1))
scaler_y = MinMaxScaler(feature_range=(-1, 1))

scaler_X.fit(X_raw[:split_idx_adv])
scaler_y.fit(y_raw[:split_idx_adv])

X_scaled = scaler_X.transform(X_raw).astype(np.float32)
y_scaled = scaler_y.transform(y_raw).astype(np.float32)

# Zaman pencerelerini oluşturma fonksiyonu (Çok değişkenli girdi için)
def make_multivariate_windows(X_data, y_data, lookback=24):
    X_win, y_win = [], []
    for i in range(lookback, len(X_data)):
        X_win.append(X_data[i - lookback:i])
        y_win.append(y_data[i, 0])
    return np.asarray(X_win, dtype=np.float32), np.asarray(y_win, dtype=np.float32).reshape(-1, 1)

# Bağlam dahil test setinin ayarlanması
test_context_X = X_scaled[split_idx_adv - LOOKBACK:]
test_context_y = y_scaled[split_idx_adv - LOOKBACK:]

X_train_adv_all, y_train_adv_all = make_multivariate_windows(X_scaled[:split_idx_adv], y_scaled[:split_idx_adv])
X_test_adv, y_test_adv = make_multivariate_windows(test_context_X, test_context_y)

# Validation setini ayırma (%10)
val_idx_adv = int(len(X_train_adv_all) * 0.90)
X_train_adv, y_train_adv = X_train_adv_all[:val_idx_adv], y_train_adv_all[:val_idx_adv]
X_val_adv, y_val_adv = X_train_adv_all[val_idx_adv:], y_train_adv_all[val_idx_adv:]

print("Çok Değişkenli Girdi Boyutu (Train):", X_train_adv.shape)
print("Çok Değişkenli Girdi Boyutu (Test):", X_test_adv.shape)

In [ ]:
# 2. Gelişmiş GRU Modelinin Tasarlanması ve Eğitilmesi
tf.keras.backend.clear_session()

advanced_model = keras.Sequential([
    keras.Input(shape=(LOOKBACK, len(feature_cols))),
    layers.GRU(64, return_sequences=True),
    layers.Dropout(0.2),
    layers.GRU(32),
    layers.Dropout(0.2),
    layers.Dense(16, activation="relu"),
    layers.Dense(1)
], name="Advanced_Multivariate_GRU")

advanced_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss="mse",
    metrics=["mae"]
)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=4, restore_best_weights=True, verbose=1
)

history_adv = advanced_model.fit(
    X_train_adv, y_train_adv,
    validation_data=(X_val_adv, y_val_adv),
    epochs=15,
    batch_size=128,
    shuffle=False,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# 3. Test tahminlerini aynı timestamp'teki gerçek fiyatlarla eşleştir
pred_scaled_adv = advanced_model.predict(X_test_adv, batch_size=128, verbose=0)
pred_ret_adv = scaler_y.inverse_transform(pred_scaled_adv).ravel()

# df_features, özellikler nedeniyle baştaki bazı satırları atmıştır; split_idx_adv bu dataframe'e aittir.
test_feature_index = df_features.index[split_idx_adv:]
y_test_prev_actual_adv = df_features["Close"].shift(1).reindex(test_feature_index).to_numpy(dtype=np.float64)
y_test_actual_adv = df_features["Close"].reindex(test_feature_index).to_numpy(dtype=np.float64)
assert len(pred_ret_adv) == len(y_test_prev_actual_adv) == len(y_test_actual_adv)
assert np.isfinite(y_test_prev_actual_adv).all() and np.isfinite(y_test_actual_adv).all()

final_price_pred = y_test_prev_actual_adv * np.exp(pred_ret_adv)
final_mae = mean_absolute_error(y_test_actual_adv, final_price_pred)
final_rmse = np.sqrt(mean_squared_error(y_test_actual_adv, final_price_pred))

print("================ TAHMİN SONUÇLARI ================")
print(f"Gelişmiş çok değişkenli GRU MAE : {final_mae:,.2f} USD")
print(f"Gelişmiş çok değişkenli GRU RMSE: {final_rmse:,.2f} USD")
print(f"Naive fiyat baseline RMSE       : {baseline_rmse:,.2f} USD")
if 'results_ret_df' in globals() and not results_ret_df.empty:
    print(f"En iyi log-getiri modeli RMSE   : {results_ret_df.iloc[0]['RMSE_USD']:,.2f} USD")
print("==================================================")

In [ ]:
# 4. Çok değişkenli GRU tahminlerini aynı test indeksinde görselleştir
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_test_actual_adv[:300], label="Gerçek kapanış", linewidth=1.5)
ax.plot(final_price_pred[:300], label="Çok değişkenli GRU", linewidth=1.2, linestyle="--")
ax.plot(y_test_prev_actual_adv[:300], label="Naive baseline (son gerçek fiyat)", alpha=0.7, linewidth=1)
ax.set_title("Çok Değişkenli GRU ve Naive Baseline — İlk 300 Test Saati")
ax.set_xlabel("Test saati")
ax.set_ylabel("BTC fiyatı (USD)")
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()

## 11. Çok Değişkenli Model Sonuçlarının Analizi

Aşağıdaki hücreler, çok değişkenli GRU'nun MAE/RMSE değerlerini tek değişkenli log-getiri modeli ve naive fiyat baseline'ı ile karşılaştırır. Önceki çalıştırmalardan kopyalanmış sabit metrikler başarı kanıtı sayılmaz.

Teknik göstergeler fiyatın geçmişinden türetildiği için bağımsız bilgi sağlamayabilir. Daha fazla özellik her zaman daha iyi sonuç vermez. Bu gözlem bizi iki soruya yönlendirir: (1) performans tek bir test penceresine mi özgü, (2) küçük hata farkı işlem maliyetlerinden sonra anlamlı kalıyor mu? Önce backtest varsayımlarını açık hale getirecek, ardından walk-forward doğrulama ekleyeceğiz.

## 12. Backtest ve Portföy Analizi

Önceki bölümde fiyat tahmini hatasını karşılaştırdık. Ancak düşük RMSE tek başına kârlı strateji anlamına gelmez. Bu bölüm, tahmin edilen log getiri yönünü basit bir stratejiye dönüştürür ve Buy & Hold ile karşılaştırır.

**Varsayımlar:** Sinyal, tahmin edilen getirinin işaretidir; pozisyon -1/+1 olur. İşlem maliyeti her pozisyon değişiminde uygulanır. Bu basit simülasyon spread, piyasa etkisi, fonlama/borçlanma maliyeti, likidite, vergi ve gerçek emir yürütmesini kapsamaz. Bu nedenle gerçek işlem sistemi değil, araştırma amaçlı bir ilk backtest'tir. Sinyal zamanlaması ve getiriler aynı test indeksinde hizalanır.

In [ ]:
# 1. Sinyalleri ve gerçekleşen getirileri hizala
pred_signals = np.where(pred_ret_adv > 0, 1.0, -1.0)
actual_test_returns = np.log(y_test_actual_adv / y_test_prev_actual_adv)
assert len(pred_signals) == len(actual_test_returns)

# İşlem maliyeti: her bir pozisyon değişiminde portföy değerinin bu oranı kadar maliyet varsayılır.
# Değeri kendi borsa/komisyon varsayımına göre değiştir. 0.001 = %0.10.
TRANSACTION_COST = 0.001
previous_position = np.r_[0.0, pred_signals[:-1]]
turnover = np.abs(pred_signals - previous_position)
transaction_costs = turnover * TRANSACTION_COST
strategy_returns = pred_signals * actual_test_returns - transaction_costs

cumulative_market_returns = np.exp(np.cumsum(actual_test_returns)) - 1.0
cumulative_strategy_returns = np.exp(np.cumsum(strategy_returns)) - 1.0

def max_drawdown_from_returns(log_returns):
    wealth = np.exp(np.cumsum(np.asarray(log_returns, dtype=float)))
    wealth = np.r_[1.0, wealth]
    peaks = np.maximum.accumulate(wealth)
    return float(np.min(wealth / peaks - 1.0))

def annualized_sharpe(log_returns, periods_per_year=365*24):
    r = np.asarray(log_returns, dtype=float)
    if len(r) < 2 or np.std(r, ddof=1) == 0:
        return np.nan
    return float(np.mean(r) / np.std(r, ddof=1) * np.sqrt(periods_per_year))

total_market_perf = cumulative_market_returns[-1] * 100
total_strategy_perf = cumulative_strategy_returns[-1] * 100
print("================ BACKTEST ÖZETİ ================")
print(f"Test başlangıç fiyatı : {y_test_prev_actual_adv[0]:,.2f} USD")
print(f"Test bitiş fiyatı     : {y_test_actual_adv[-1]:,.2f} USD")
print(f"Buy & Hold getirisi   : %{total_market_perf:.2f}")
print(f"Strateji net getirisi : %{total_strategy_perf:.2f}")
print(f"Varsayılan işlem maliyeti: %{TRANSACTION_COST*100:.3f} / pozisyon değişimi")
print(f"Pozisyon değişimi     : {int(np.sum(turnover > 0))}")
print(f"Strateji Sharpe (yıllıklandırılmış): {annualized_sharpe(strategy_returns):.3f}")
print(f"Strateji maksimum düşüş: %{max_drawdown_from_returns(strategy_returns)*100:.2f}")
print("================================================")

In [ ]:
# 2. Kümülatif performansı karşılaştır
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(cumulative_market_returns * 100, label="Buy & Hold", alpha=0.85)
ax.plot(cumulative_strategy_returns * 100, label="GRU stratejisi (maliyet sonrası)", linewidth=1.5)
ax.set_title("Model Stratejisi ve Buy & Hold — Kümülatif Getiri")
ax.set_xlabel("Test saati")
ax.set_ylabel("Kümülatif getiri (%)")
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend()
plt.show()

## 13. Walk-Forward Doğrulama, Güçlü Baseline'lar ve Kalite Kapıları

Tek bir kronolojik test bölümü, performansın yalnızca belirli bir piyasa rejimine bağlı olup olmadığını göstermez. Walk-forward değerlendirmede her fold için geçmiş veriyle eğitim, sonraki blokta test yapılır. Önce naive ve tarihsel ortalama-getiri benchmark'larını birden fazla fold üzerinde ölçer, ardından çok değişkenli GRU'yu her fold için yalnızca geçmiş verilerle yeniden eğitiriz. Ölçekleyiciler de her fold'da yalnızca o fold'un eğitim verisine fit edilir. Bu bölüm hesaplama açısından daha ağırdır; Colab oturum süresini yönetmek için fold ve epoch sayıları ayarlanabilir.

Ek olarak veri hizalaması, eksik değer, sonlu metrik ve zaman sıralaması kontrolleri çalışır.

In [ ]:
# 13.1 Zaman indeksine dayalı kalite kapıları
assert hourly.index.is_monotonic_increasing, "Saatlik veri kronolojik sıralı değil."
assert not hourly.index.has_duplicates, "Saatlik indekste yinelenen zaman damgaları var."
assert (hourly["Close"] > 0).all(), "Kapanış fiyatlarında sıfır/negatif değer var."
assert np.isfinite(hourly["Close"].to_numpy(dtype=float)).all(), "Kapanış fiyatlarında NaN/inf var."
assert len(X_train) > 0 and len(X_val) > 0 and len(X_test) > 0
assert len(y_test_actual) == len(y_naive)
assert len(y_test_actual_adv) == len(final_price_pred) == len(pred_ret_adv)
print("Kalite kapıları geçti: kronoloji, benzersiz indeks, pozitif/sonlu fiyat ve tahmin hizalaması.")

In [ ]:
# 13.2 Walk-forward benchmark: naive fiyat ve geçmiş ortalama log-getiri
# Fold'lar saatlik serinin son kısmında, kronolojik ve çakışmayan test blokları üretir.
# Her fold'un tahminleri yalnızca fold başlangıcından önceki geçmişten hesaplanır.
N_FOLDS = 4
TEST_BLOCK_SIZE = max(24, len(hourly) // 25)  # veri büyüklüğüne göre yaklaşık %4'lük blok
MIN_TRAIN_SIZE = max(LOOKBACK * 10, len(hourly) // 2)
fold_rows = []
close_series = hourly["Close"].astype(float)
log_returns_series = np.log(close_series / close_series.shift(1))

for fold in range(N_FOLDS):
    test_end = len(hourly) - (N_FOLDS - 1 - fold) * TEST_BLOCK_SIZE
    test_start = test_end - TEST_BLOCK_SIZE
    if test_start < MIN_TRAIN_SIZE or test_end > len(hourly):
        continue
    train_close = close_series.iloc[:test_start]
    test_close = close_series.iloc[test_start:test_end]
    previous_close = close_series.iloc[test_start - 1:test_end - 1].to_numpy()
    actual_close = test_close.to_numpy()
    # Benchmark 1: random-walk/naive, sonraki kapanış = önceki kapanış
    naive_pred = previous_close
    # Benchmark 2: eğitim dönemindeki ortalama saatlik log getiriyi kullan
    train_mean_return = float(log_returns_series.iloc[1:test_start].mean())
    mean_return_pred = previous_close * np.exp(train_mean_return)
    fold_rows.append({
        "Fold": fold + 1,
        "Train_end": str(train_close.index[-1]),
        "Test_start": str(test_close.index[0]),
        "Test_end": str(test_close.index[-1]),
        "Naive_MAE_USD": mean_absolute_error(actual_close, naive_pred),
        "Naive_RMSE_USD": np.sqrt(mean_squared_error(actual_close, naive_pred)),
        "MeanReturn_MAE_USD": mean_absolute_error(actual_close, mean_return_pred),
        "MeanReturn_RMSE_USD": np.sqrt(mean_squared_error(actual_close, mean_return_pred)),
        "Test_records": len(test_close)
    })

walk_forward_df = pd.DataFrame(fold_rows)
if walk_forward_df.empty:
    print("Walk-forward için yeterli kayıt bulunamadı. Veri seti/fold ayarlarını kontrol edin.")
else:
    display(walk_forward_df)
    print("Fold ortalamaları:")
    display(walk_forward_df[["Naive_MAE_USD", "Naive_RMSE_USD", "MeanReturn_MAE_USD", "MeanReturn_RMSE_USD"]].mean().to_frame("Ortalama"))

### 13.3 Walk-forward GRU eğitimi

Benchmark'lar, tek bir test penceresine bağlı olmayan referans sağlar. Şimdi aynı mantığı GRU'ya uygularız: her fold'da scaler ve model yalnızca geçmiş veriden öğrenir, sonraki blokta değerlendirilir. Her fold'un naive baseline ile aynı timestamp'lerde karşılaştırılması, modelin farklı dönemlerde ne kadar tutarlı olduğunu gösterir.

In [ ]:
# 13.3 Gerçek walk-forward GRU: her fold'da baştan eğitim
# Colab süresi kısıtlıysa RUN_WALK_FORWARD_GRU=False yaparak yalnızca benchmark fold'larını çalıştırabilirsin.
RUN_WALK_FORWARD_GRU = True
WF_GRU_FOLDS = 3
WF_GRU_EPOCHS = 8
WF_GRU_BATCH_SIZE = 128
wf_gru_rows = []

if RUN_WALK_FORWARD_GRU:
    X_wf_raw = df_features[feature_cols].to_numpy(dtype=np.float32)
    y_wf_raw = df_features[target_col].to_numpy(dtype=np.float32).reshape(-1, 1)
    n_wf = len(df_features)
    wf_test_size = max(24, n_wf // 20)
    wf_min_train = max(LOOKBACK * 20, n_wf // 2)

    for wf_fold in range(WF_GRU_FOLDS):
        wf_test_end = n_wf - (WF_GRU_FOLDS - 1 - wf_fold) * wf_test_size
        wf_test_start = wf_test_end - wf_test_size
        if wf_test_start < wf_min_train or wf_test_end > n_wf:
            continue

        # Fold'a özel scaler'lar sadece fold eğitim aralığında fit edilir.
        wf_scaler_X = MinMaxScaler(feature_range=(-1, 1))
        wf_scaler_y = MinMaxScaler(feature_range=(-1, 1))
        wf_scaler_X.fit(X_wf_raw[:wf_test_start])
        wf_scaler_y.fit(y_wf_raw[:wf_test_start])
        wf_X_scaled = wf_scaler_X.transform(X_wf_raw).astype(np.float32)
        wf_y_scaled = wf_scaler_y.transform(y_wf_raw).astype(np.float32)

        X_wf_train_all, y_wf_train_all = make_multivariate_windows(
            wf_X_scaled[:wf_test_start], wf_y_scaled[:wf_test_start], LOOKBACK
        )
        wf_val_idx = int(len(X_wf_train_all) * 0.90)
        if wf_val_idx <= 0 or wf_val_idx >= len(X_wf_train_all):
            print(f"Fold {wf_fold+1}: eğitim/validasyon verisi yetersiz; atlandı.")
            continue
        X_wf_train, y_wf_train = X_wf_train_all[:wf_val_idx], y_wf_train_all[:wf_val_idx]
        X_wf_val, y_wf_val = X_wf_train_all[wf_val_idx:], y_wf_train_all[wf_val_idx:]

        X_wf_test_context = wf_X_scaled[wf_test_start-LOOKBACK:wf_test_end]
        y_wf_test_context = wf_y_scaled[wf_test_start-LOOKBACK:wf_test_end]
        X_wf_test, _ = make_multivariate_windows(X_wf_test_context, y_wf_test_context, LOOKBACK)

        tf.keras.backend.clear_session()
        wf_model = keras.Sequential([
            keras.Input(shape=(LOOKBACK, len(feature_cols))),
            layers.GRU(64, return_sequences=True),
            layers.Dropout(0.2),
            layers.GRU(32),
            layers.Dropout(0.2),
            layers.Dense(16, activation="relu"),
            layers.Dense(1)
        ], name=f"WalkForward_GRU_Fold_{wf_fold+1}")
        wf_model.compile(optimizer=keras.optimizers.Adam(learning_rate=5e-4), loss="mse")
        wf_early_stop = keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True, verbose=0
        )
        wf_model.fit(
            X_wf_train, y_wf_train,
            validation_data=(X_wf_val, y_wf_val),
            epochs=WF_GRU_EPOCHS,
            batch_size=WF_GRU_BATCH_SIZE,
            shuffle=False,
            callbacks=[wf_early_stop],
            verbose=0
        )
        wf_pred_scaled = wf_model.predict(X_wf_test, batch_size=WF_GRU_BATCH_SIZE, verbose=0)
        wf_pred_returns = wf_scaler_y.inverse_transform(wf_pred_scaled).ravel()

        wf_idx = df_features.index[wf_test_start:wf_test_end]
        wf_actual = df_features["Close"].reindex(wf_idx).to_numpy(dtype=np.float64)
        wf_previous = df_features["Close"].shift(1).reindex(wf_idx).to_numpy(dtype=np.float64)
        wf_pred_prices = wf_previous * np.exp(wf_pred_returns)
        wf_naive_prices = wf_previous
        wf_gru_rows.append({
            "Fold": wf_fold + 1,
            "Train_end": str(df_features.index[wf_test_start-1]),
            "Test_start": str(wf_idx[0]),
            "Test_end": str(wf_idx[-1]),
            "Records": len(wf_idx),
            "GRU_MAE_USD": mean_absolute_error(wf_actual, wf_pred_prices),
            "GRU_RMSE_USD": np.sqrt(mean_squared_error(wf_actual, wf_pred_prices)),
            "Naive_MAE_USD": mean_absolute_error(wf_actual, wf_naive_prices),
            "Naive_RMSE_USD": np.sqrt(mean_squared_error(wf_actual, wf_naive_prices)),
        })
        print(f"Walk-forward fold {wf_fold+1}/{WF_GRU_FOLDS} tamamlandı.")

    wf_gru_df = pd.DataFrame(wf_gru_rows)
    if wf_gru_df.empty:
        print("Walk-forward GRU fold'u üretilemedi; veri uzunluğu ve fold ayarlarını kontrol et.")
    else:
        display(wf_gru_df)
        print("Fold ortalamaları:")
        display(wf_gru_df[["GRU_MAE_USD", "GRU_RMSE_USD", "Naive_MAE_USD", "Naive_RMSE_USD"]].mean().to_frame("Ortalama"))
        print("GRU fold'larının baseline'ı geçtiği oran:",
              f"{(wf_gru_df['GRU_RMSE_USD'] < wf_gru_df['Naive_RMSE_USD']).mean():.1%}")
else:
    print("Walk-forward GRU eğitimi kapalı; yalnızca benchmark fold sonuçları raporlanır.")

In [ ]:
# 13.4 Aynı timestamp'lerde adil model karşılaştırması
# Farklı ön işlem adımları test başlangıcını birkaç saat kaydırabilir.
# Bu nedenle modelleri yalnızca ortak test timestamp'lerinde karşılaştırıyoruz.
common_idx = test_feature_index.intersection(test_return_index)
if len(common_idx) == 0:
    raise ValueError("Modeller arasında ortak test timestamp'i bulunamadı.")

common_actual = hourly.loc[common_idx, "Close"].to_numpy(dtype=np.float64)
common_previous = hourly["Close"].shift(1).reindex(common_idx).to_numpy(dtype=np.float64)
comparison_rows = []
comparison_rows.append({
    "Model": "Naive fiyat baseline",
    "MAE_USD": mean_absolute_error(common_actual, common_previous),
    "RMSE_USD": np.sqrt(mean_squared_error(common_actual, common_previous)),
})

# Çok değişkenli GRU tahminlerini timestamp indeksine bağla.
advanced_pred_series = pd.Series(final_price_pred, index=test_feature_index)
advanced_common_pred = advanced_pred_series.reindex(common_idx).to_numpy(dtype=np.float64)
comparison_rows.append({
    "Model": "Çok değişkenli GRU",
    "MAE_USD": mean_absolute_error(common_actual, advanced_common_pred),
    "RMSE_USD": np.sqrt(mean_squared_error(common_actual, advanced_common_pred)),
})

# Tek değişkenli log-getiri modellerinin tahminlerini aynı ortak timestamp'lerde fiyat ölçeğine çevir.
for model_name, predicted_returns in models_ret.items():
    return_series = pd.Series(predicted_returns, index=test_return_index)
    common_predicted_returns = return_series.reindex(common_idx).to_numpy(dtype=np.float64)
    predicted_prices = common_previous * np.exp(common_predicted_returns)
    comparison_rows.append({
        "Model": f"{model_name} (LogRet)",
        "MAE_USD": mean_absolute_error(common_actual, predicted_prices),
        "RMSE_USD": np.sqrt(mean_squared_error(common_actual, predicted_prices)),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values("RMSE_USD").reset_index(drop=True)
assert np.isfinite(comparison_df[["MAE_USD", "RMSE_USD"]].to_numpy()).all(), "Karşılaştırma tablosunda NaN/inf metrik var."
display(comparison_df)
print("Ortak test timestamp sayısı:", len(common_idx))
print("En düşük RMSE (yalnızca bu ortak test aralığında):", comparison_df.iloc[0]["Model"])
print("Not: Tahmin hatası tek başına kârlılık kanıtı değildir; maliyet sonrası backtest ve walk-forward tutarlılığı ayrıca değerlendirilmelidir.")

## Son değerlendirme ve bir sonraki iyileştirme döngüsü

Bu sürümün ana kazanımı, daha fazla katman eklemek değil; iddiaları test edilebilir hale getirmek, sabit hizalama sayılarından kurtulmak, sonuçları doğru timestamp'lerde karşılaştırmak ve işlem maliyetini görünür kılmaktır.

### Bu sürümden sonra kendimize soracağımız sorular
1. GRU, her walk-forward fold'unda naive baseline'ı geçiyor mu, yoksa tek bir dönemde mi iyi görünüyor?
2. Fiyat RMSE'sindeki küçük bir iyileşme, işlem maliyeti sonrası pozitif ve istikrarlı getiriye dönüşüyor mu?
3. RSI/SMA/volatilite özellikleri ablation testinde gerçekten katkı sağlıyor mu?
4. Sonuçlar farklı lookback süreleri, farklı piyasa rejimleri ve farklı rastgele tohumlarda korunuyor mu?
5. Daha yüksek frekanslı veri kullanacaksak spread, likidite ve işlem gecikmesi gibi uygulama kısıtlarını modelleyebiliyor muyuz?

### Önerilen sonraki sürüm
Bir sonraki kalite adımı, her fold içinde GRU'yu yeniden eğitip naive ve ortalama-getiri benchmark'larıyla karşılaştırmak; ardından özellik ablation'ı, farklı seed'ler ve maliyet duyarlılık analizi eklemektir. Bu sürümde sonuçlar doğrulanmadan kârlılık iddiası üretilmez.